# AgroPest-12 Part B: Standalone I-JEPA Pretraining for YOLOv12m

**Model:** random `yolo12m.yaml` backbone  
**Input:** `ssl_pool/images` only, 9,180 images  
**Runtime:** Kaggle T4 x2 through PyTorch Distributed Data Parallel (DDP)

This notebook implements I-JEPA directly with PyTorch and TorchVision. It does not install, import, or depend on `ssl-detection-lab`, `ssldet`, Lightly, or PyTorch Lightning. The online YOLOv12m backbone is trained from random YAML initialization and is the only learned component transferred to Notebook 03b.

## Protocol And CNN Adaptation

The EMA target encoder sees the complete image and produces P4 (stride-16) and P5 (stride-32) feature grids. Four rectangular target blocks are sampled on P4. Their union is upsampled and masked in the online encoder input **before** the YOLO CNN runs, so the context branch cannot trivially use an unmasked target pixel. Independent Transformer predictors use visible context tokens plus learned mask tokens and unique 2D positional encodings to predict target latents at both levels.

This is a YOLO-CNN adaptation of I-JEPA, not a reproduction of the original ViT-H/14, 300-epoch ImageNet experiment. It keeps the method's essential latent prediction, target masking, EMA teacher, positional target queries, and temporary predictor while using the actual routed YOLO backbone required for detector transfer. The notebook constructs only `ssl_pool/images`; labels, validation images, and test images are never constructed or read.

In [ ]:
# Kaggle already provides CUDA-matched PyTorch. Do not install another torch wheel.
%pip install -q --upgrade "ultralytics>=8.4.96" "scikit-learn>=1.5" seaborn

## Configuration

Every value is explicit and commented for the viva. The original I-JEPA configuration uses ViT-H/14, 16 A100 GPUs, a global batch of 2,048, and 300 epochs, so those absolute settings are not transferable. Here 256px produces a 16x16 P4 grid, matching the original ViT-H/14 token-grid resolution at 224px, while a conservative source batch of 16 per T4 leaves memory for both CNN encoders and the predictor.

In [ ]:
from dataclasses import asdict, dataclass
from pathlib import Path
import shutil
import json


@dataclass
class CFG:
    # Dataset and reproducibility
    partb_root: str = '/kaggle/input/datasets/sadikshahriar/agropest-12-partb-safe/AgroPest-12-PartB-Safe'  # Published leakage-safe Part B dataset.
    output_dir: str = '/kaggle/working/partB_03a_ijepa_yolo12m'  # Writable Kaggle output; publish this complete directory.
    seed: int = 445  # Shared Part B seed for DDP shuffling, masks, and diagnostics.

    # Strict label-free YOLO encoder and runtime
    model_yaml: str = 'yolo12m.yaml'  # Random weights only; yolo12m.pt is reserved for the downstream COCO baseline.
    image_size: int = 256  # Gives the selected stride-16 P4 feature level a 16x16 grid, like ViT-H/14 at 224px.
    p4_stride: int = 16  # P4 provides a 16x16 grid that preserves spatial locality.
    p5_stride: int = 32  # P5 is included in the SSL loss so every transferable YOLO backbone layer receives gradients.
    epochs: int = 30  # Fixed Part B SSL budget; this is a short domain adaptation, not paper-equivalent training.
    per_gpu_batch: int = 16  # Conservative source-image batch per T4 for online CNN, EMA CNN, and token predictor.
    workers: int = 2  # Conservative workers per DDP rank for Kaggle CPU resources.
    amp: bool = True  # FP16 autocast and GradScaler improve T4 throughput without replacing Kaggle PyTorch.
    amp_init_scale: float = 4096.0  # Conservative initial loss scale avoids FP16 gradient overflow from random SSL heads.
    amp_growth_interval: int = 10000  # Longer than this run; prevents the scale growing unnecessarily after stable early epochs.
    max_consecutive_amp_skips: int = 8  # Transient overflow is recovered by GradScaler; repeated overflow stops a genuinely unstable run.

    # AdamW and schedules
    learning_rate: float = 2e-5  # Conservative scaling of the official 1e-3 LR from global batch 2,048 to global batch 32.
    min_learning_rate: float = 1e-6  # Non-zero cosine-decay floor for the short run.
    warmup_epochs: int = 4  # Roughly the official 40/300 warmup proportion, protecting random CNN startup.
    warmup_start_factor: float = 0.10  # Starts warmup at 10% of peak LR rather than zero.
    weight_decay_start: float = 0.04  # Official I-JEPA AdamW decay at the beginning of training.
    weight_decay_end: float = 0.40  # Cosine increase used by the official I-JEPA recipe; norm and bias parameters are exempt.
    gradient_clip_norm: float = 1.0  # Practical finite-gradient guard for random CNN SSL after AMP unscale.

    # I-JEPA masking and predictor
    num_target_blocks: int = 4  # Official multi-block target count; four regions provide dense spatial prediction signal.
    target_scale_min: float = 0.15  # Official target-block area fraction of the P4 grid.
    target_scale_max: float = 0.20  # Keeps individual semantic targets neither tiny nor nearly global.
    target_aspect_min: float = 0.75  # Official target rectangle aspect-ratio range.
    target_aspect_max: float = 1.50
    min_context_cells: int = 10  # Rejects sampled masks that leave too little visible P4 or P5 context.
    predictor_dim: int = 256  # Smaller than the 512-channel YOLO P4 map; the predictor is discarded after SSL.
    predictor_depth: int = 4  # Retains multi-layer token interaction while fitting a short two-T4 run.
    predictor_heads: int = 8  # 32-dimensional attention heads at predictor_dim=256.
    predictor_mlp_ratio: float = 4.0  # Conventional Transformer feed-forward expansion.
    p4_loss_weight: float = 0.5  # Equal multi-scale contribution preserves detailed P4 supervision.
    p5_loss_weight: float = 0.5  # Forces the final YOLO backbone stage to learn transferable latent features.
    base_momentum: float = 0.996  # Official teacher EMA start; it rises linearly per successful optimizer step.

    # Image-only anti-collapse diagnostics and exports
    probe_images: int = 64  # Fixed SSL-pool probe used only for per-epoch feature-health checks.
    min_feature_std: float = 1e-3  # Rejects near-constant target features like the failed library run.
    max_cosine_p95: float = 0.9995  # Rejects a probe where almost every image representation is identical.
    diagnostic_images: int = 500  # Fixed SSL-pool sample for t-SNE and retrieval.
    diagnostic_batch_size: int = 64
    retrieval_queries: int = 4
    retrieval_neighbors: int = 5
    tsne_perplexity: float = 30.0
    tsne_iterations: int = 1000
    figure_dpi: int = 180
    resume: bool = False  # Set True only to continue the identical configuration from last_ssl.pt.


cfg = CFG()
PARTB_ROOT = Path(cfg.partb_root)
SSL_IMAGES = PARTB_ROOT / 'ssl_pool' / 'images'
OUTPUT_DIR = Path(cfg.output_dir)
if OUTPUT_DIR.exists() and not cfg.resume: shutil.rmtree(OUTPUT_DIR)  # Fresh runs must not reuse failed-run history or checkpoints.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG = asdict(cfg)
(OUTPUT_DIR / 'ijepa_config.json').write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
print(json.dumps(CONFIG, indent=2))

### I-JEPA Design Rationale

At 256px, the YOLOv12m stride-16 P4 level is a 16x16 feature grid: enough spatial context for latent targets without the 1,024-token cost of P3. The simultaneous P5 objective ensures final backbone layers also receive SSL gradients before full-backbone transfer. A source batch of 16 per T4 leaves memory for the online CNN, EMA CNN, and two compact token predictors. Four 15-20% target blocks create distributed semantic targets while retaining visible context. LR 2e-5 is conservative linear scaling from the official 1e-3/global-2,048 recipe to global batch 32; four warmup epochs stabilize random-CNN startup. EMA ramps from 0.996 to 1.0. The image-only feature-variance and cosine guard rejects the near-constant representation collapse observed in the previous library run.

The original I-JEPA evidence used ViT-H/14, ImageNet, and 300 epochs, so this is not a direct reproduction: [CVPR paper](https://openaccess.thecvf.com/content/CVPR2023/papers/Assran_Self-Supervised_Learning_From_Images_With_a_Joint-Embedding_Predictive_Architecture_CVPR_2023_paper.pdf), [official config](https://github.com/facebookresearch/ijepa/blob/main/configs/in1k_vith14_ep300.yaml).

## Standalone I-JEPA Core

The helper module below is generated into `/kaggle/working/` so `torch.distributed.run` can import it in two clean worker processes. `YOLOPyramidEncoder` executes the YAML-defined backbone using each module's `f`, `i`, and `save` routing attributes; it does not replace the backbone with a fixed sequential slice. P4 and P5 both feed latent-prediction losses, so all transferable backbone stages receive gradients. Each predictor adds positional encodings **after** inserting mask tokens, preventing the positional-token bug found in the previous package.

In [ ]:
from textwrap import dedent

CORE_MODULE = OUTPUT_DIR / 'ijepa_core.py'
CORE_MODULE.write_text(dedent(r'''
import copy
import csv
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.distributed import DistributedSampler
from torchvision import transforms
from ultralytics import YOLO

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True

def list_ssl_images(image_dir):
    paths = sorted(path for path in Path(image_dir).iterdir() if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES)
    if len(paths) != 9180: raise ValueError(f'Expected 9,180 SSL-pool images, found {len(paths)}')
    return paths

def build_ijepa_transform(config, train=True):
    if train:
        return transforms.Compose([transforms.RandomResizedCrop(config['image_size'], scale=(0.3, 1.0), ratio=(0.75, 1.333333)), transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    return transforms.Compose([transforms.Resize(config['image_size'] + 32), transforms.CenterCrop(config['image_size']), transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

class ImageOnlyDataset(Dataset):
    def __init__(self, paths, transform): self.paths, self.transform = list(paths), transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image: return self.transform(image.convert('RGB'))

class YOLOPyramidEncoder(nn.Module):
    def __init__(self, task_model, p4_stride, p5_stride):
        super().__init__()
        backbone = getattr(task_model, 'yaml', {}).get('backbone', [])
        if not backbone: raise ValueError('The Ultralytics model has no YAML backbone definition.')
        self.layers = nn.ModuleList(list(task_model.model[:len(backbone)]))
        self.save = set(getattr(task_model, 'save', []))
        self.p4_stride, self.p5_stride = p4_stride, p5_stride
    def forward(self, x):
        p4_shape = (x.shape[-2] // self.p4_stride, x.shape[-1] // self.p4_stride)
        p5_shape = (x.shape[-2] // self.p5_stride, x.shape[-1] // self.p5_stride)
        saved, p4, p5 = [], None, None
        for module in self.layers:
            source = getattr(module, 'f', -1)
            if source != -1:
                x = saved[source] if isinstance(source, int) else [x if item == -1 else saved[item] for item in source]
            x = module(x)
            if isinstance(x, torch.Tensor) and x.shape[-2:] == p4_shape: p4 = x
            if isinstance(x, torch.Tensor) and x.shape[-2:] == p5_shape: p5 = x
            saved.append(x if getattr(module, 'i', -1) in self.save else None)
        if p4 is None or p5 is None: raise RuntimeError('Could not find both P4 and P5 tensors in the routed YOLO backbone.')
        return {'p4': p4, 'p5': p5}

def sine_position_encoding(height, width, dim, device):
    if dim % 4: raise ValueError('Predictor dimension must be divisible by four.')
    y, x = torch.meshgrid(torch.arange(height, device=device), torch.arange(width, device=device), indexing='ij')
    omega = torch.arange(dim // 4, device=device, dtype=torch.float32) / (dim // 4)
    omega = 1.0 / (10000 ** omega)
    y, x = y.reshape(-1, 1) * omega, x.reshape(-1, 1) * omega
    return torch.cat((y.sin(), y.cos(), x.sin(), x.cos()), dim=1).unsqueeze(0)

def sample_target_masks(batch, height, width, config, device):
    masks = torch.zeros(batch, config['num_target_blocks'], height, width, dtype=torch.bool, device=device)
    cells = height * width
    for sample in range(batch):
        for _ in range(100):
            masks[sample].zero_()
            for block in range(config['num_target_blocks']):
                area = random.uniform(config['target_scale_min'], config['target_scale_max']) * cells
                aspect = random.uniform(config['target_aspect_min'], config['target_aspect_max'])
                block_h = min(height, max(1, round(math.sqrt(area / aspect))))
                block_w = min(width, max(1, round(math.sqrt(area * aspect))))
                top, left = random.randint(0, height - block_h), random.randint(0, width - block_w)
                masks[sample, block, top:top + block_h, left:left + block_w] = True
            p4_union = masks[sample].any(dim=0)
            p5_union = F.adaptive_max_pool2d(p4_union.float().unsqueeze(0).unsqueeze(0), (height // 2, width // 2)).squeeze().bool()
            if (~p4_union).sum() >= config['min_context_cells'] and (~p5_union).sum() >= config['min_context_cells']: break
        else: raise RuntimeError('Could not sample masks with enough P4 and P5 context cells.')
    return masks

class LatentPredictor(nn.Module):
    def __init__(self, input_dim, config):
        super().__init__()
        dim = config['predictor_dim']
        self.input_proj, self.output_proj = nn.Linear(input_dim, dim), nn.Linear(dim, input_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, dim))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        layer = nn.TransformerEncoderLayer(d_model=dim, nhead=config['predictor_heads'], dim_feedforward=int(dim * config['predictor_mlp_ratio']), dropout=0.0, activation='gelu', batch_first=True, norm_first=True)
        self.transformer, self.norm = nn.TransformerEncoder(layer, num_layers=config['predictor_depth']), nn.LayerNorm(dim)
    def forward(self, context_map, target_union):
        batch, _, height, width = context_map.shape
        tokens = self.input_proj(context_map.flatten(2).transpose(1, 2))
        target_flat = target_union.flatten(1)
        token_content = torch.where(target_flat.unsqueeze(-1), self.mask_token.expand(batch, tokens.shape[1], -1), tokens)
        position = sine_position_encoding(height, width, tokens.shape[-1], tokens.device).to(tokens.dtype)
        return self.output_proj(self.norm(self.transformer(token_content + position)))

class IJEPA(nn.Module):
    def __init__(self, encoder, config, feature_dims):
        super().__init__()
        self.encoder, self.teacher = encoder, copy.deepcopy(encoder)
        self.predictors = nn.ModuleDict({level: LatentPredictor(feature_dim, config) for level, feature_dim in feature_dims.items()})
        for parameter in self.teacher.parameters(): parameter.requires_grad_(False)
    def forward(self, images, target_masks):
        p4_union = target_masks.any(dim=1)
        pixel_mask = F.interpolate(p4_union.float().unsqueeze(1), size=images.shape[-2:], mode='nearest').bool()
        context_images = images.masked_fill(pixel_mask, 0.0)
        context_maps = self.encoder(context_images)
        self.teacher.eval()  # Freeze teacher BatchNorm statistics; they are copied from the online encoder after valid steps.
        with torch.no_grad(): teacher_maps = self.teacher(images)
        target_unions = {'p4': p4_union, 'p5': F.adaptive_max_pool2d(p4_union.float().unsqueeze(1), teacher_maps['p5'].shape[-2:]).squeeze(1).bool()}
        predictions = {level: self.predictors[level](context_maps[level], target_unions[level]) for level in target_unions}
        targets = {level: F.layer_norm(teacher_maps[level].flatten(2).transpose(1, 2).float(), (teacher_maps[level].shape[1],)) for level in target_unions}
        return predictions, targets, target_unions, context_maps, teacher_maps
    @torch.no_grad()
    def update_teacher(self, momentum):
        for student, teacher in zip(self.encoder.parameters(), self.teacher.parameters()): teacher.mul_(momentum).add_(student, alpha=1.0 - momentum)
        for student, teacher in zip(self.encoder.buffers(), self.teacher.buffers()): teacher.copy_(student)

def cosine_schedule(start, end, step, total):
    progress = min(step / max(1, total - 1), 1.0)
    return end + (start - end) * 0.5 * (1.0 + math.cos(math.pi * progress))

def learning_rate(config, step, total, warmup):
    if step < warmup: return config['learning_rate'] * (config['warmup_start_factor'] + (1.0 - config['warmup_start_factor']) * (step + 1) / max(1, warmup))
    return cosine_schedule(config['learning_rate'], config['min_learning_rate'], step - warmup, total - warmup)

def momentum_for_step(config, step, total): return config['base_momentum'] + (1.0 - config['base_momentum']) * min(step / max(1, total - 1), 1.0)

def parameter_groups(module, decay):
    with_decay, without_decay = [], []
    for name, parameter in module.named_parameters():
        if parameter.requires_grad: (without_decay if parameter.ndim == 1 or name.endswith('.bias') else with_decay).append(parameter)
    return [{'params': with_decay, 'weight_decay': decay}, {'params': without_decay, 'weight_decay': 0.0}]

def health_metrics(teacher_map, predictions, targets, target_union):
    pooled = teacher_map.float().mean((2, 3))
    normalized = F.normalize(pooled, dim=1)
    cosine = normalized @ normalized.T
    off_diagonal = cosine[~torch.eye(len(pooled), dtype=torch.bool, device=pooled.device)]
    target_flat = target_union.flatten(1)
    prediction_cosine = F.cosine_similarity(predictions[target_flat].float(), targets[target_flat].float(), dim=1).mean()
    return torch.stack((pooled.std(dim=0).mean(), off_diagonal.mean(), torch.quantile(off_diagonal, 0.95), prediction_cosine))

def gradients_are_finite(parameters): return all(parameter.grad is None or torch.isfinite(parameter.grad).all() for parameter in parameters)

def cpu_state(module): return {key: value.detach().cpu() for key, value in module.state_dict().items()}

def save_state(output_dir, epoch, best_loss, global_step, model, optimizer, scaler, config, health):
    state = {'format_version': 3, 'epoch': epoch, 'best_loss': best_loss, 'global_step': global_step, 'student_encoder_state_dict': cpu_state(model.encoder), 'teacher_encoder_state_dict': cpu_state(model.teacher), 'predictors_state_dict': cpu_state(model.predictors), 'optimizer_state_dict': optimizer.state_dict(), 'scaler_state_dict': scaler.state_dict(), 'rng_state': torch.get_rng_state(), 'cuda_rng_state': torch.cuda.get_rng_state(), 'config': config, 'health': health}
    torch.save(state, output_dir / 'last_ssl.pt')
    return state

def export_backbone(output_dir, config):
    state = torch.load(output_dir / 'best_ssl.pt', map_location='cpu', weights_only=False)
    detector = YOLO(config['model_yaml'])
    encoder = YOLOPyramidEncoder(detector.model, config['p4_stride'], config['p5_stride'])
    source = state['student_encoder_state_dict']; target = encoder.state_dict()
    matched = [key for key, value in source.items() if key in target and target[key].shape == value.shape]
    if len(matched) != len(source) or len(matched) != len(target): raise RuntimeError(f'Backbone coverage must be 100%, got {len(matched)}/{len(source)} source and {len(matched)}/{len(target)} target keys.')
    encoder.load_state_dict(source, strict=True)
    backbone_path = output_dir / 'ijepa_best_backbone_yolo12m.pt'
    detector_path = output_dir / 'ijepa_best_backbone_yolo12m_detector.pt'
    torch.save({'format_version': 3, 'yolo_model': config['model_yaml'], 'p4_stride': config['p4_stride'], 'p5_stride': config['p5_stride'], 'encoder_state_dict': cpu_state(encoder), 'source_ssl_checkpoint': str(output_dir / 'best_ssl.pt')}, backbone_path)
    detector.model.args = {'model': config['model_yaml'], 'task': 'detect'}
    torch.save({'model': copy.deepcopy(detector.model).half(), 'ema': None, 'optimizer': None, 'epoch': state['epoch'], 'train_args': detector.model.args}, detector_path)
    report = {'source': str(output_dir / 'best_ssl.pt'), 'target_model': config['model_yaml'], 'backbone_key_coverage': 1.0, 'matched_keys': len(matched), 'target_keys': len(target), 'student_context_encoder_transferred': True, 'teacher_transferred': False, 'predictor_transferred': False, 'detector_neck_head_transferred': False}
    (output_dir / 'transfer_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    return report

def train_distributed(config):
    local_rank, rank, world_size = int(os.environ['LOCAL_RANK']), int(os.environ['RANK']), int(os.environ['WORLD_SIZE'])
    torch.cuda.set_device(local_rank); device = torch.device('cuda', local_rank); dist.init_process_group(backend='nccl')
    seed_everything(config['seed'] + rank); output_dir = Path(config['output_dir'])
    paths = list_ssl_images(Path(config['partb_root']) / 'ssl_pool' / 'images')
    dataset = ImageOnlyDataset(paths, build_ijepa_transform(config, train=True))
    probe_paths = sorted(random.Random(config['seed']).sample(paths, min(config['probe_images'], len(paths))))
    probe_loader = DataLoader(ImageOnlyDataset(probe_paths, build_ijepa_transform(config, train=False)), batch_size=len(probe_paths), shuffle=False, num_workers=config['workers'], pin_memory=True)
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True, seed=config['seed'], drop_last=True)
    loader = DataLoader(dataset, batch_size=config['per_gpu_batch'], sampler=sampler, num_workers=config['workers'], pin_memory=True, persistent_workers=config['workers'] > 0, drop_last=True)
    detector = YOLO(config['model_yaml']).model.to(device)
    encoder = YOLOPyramidEncoder(detector, config['p4_stride'], config['p5_stride']).to(device)
    with torch.no_grad(): feature_dims = {level: value.shape[1] for level, value in encoder(torch.zeros(1, 3, config['image_size'], config['image_size'], device=device)).items()}
    # Both P4 and P5 predictions contribute to loss, so every trainable backbone parameter participates in DDP reduction.
    model = DDP(IJEPA(encoder, config, feature_dims).to(device), device_ids=[local_rank], output_device=local_rank, broadcast_buffers=True)
    optimizer = torch.optim.AdamW(parameter_groups(model.module, config['weight_decay_start']), lr=config['learning_rate'], betas=(0.9, 0.999), eps=1e-8)
    scaler = torch.amp.GradScaler('cuda', enabled=config['amp'], init_scale=config['amp_init_scale'], growth_interval=config['amp_growth_interval'])
    history_path, last_path, history = output_dir / 'history.csv', output_dir / 'last_ssl.pt', []
    best_loss, global_step, start_epoch, consecutive_amp_skips = float('inf'), 0, 0, 0
    if config['resume'] and last_path.is_file():
        state = torch.load(last_path, map_location='cpu', weights_only=False)
        if state.get('format_version') != 3: raise ValueError('This multi-scale run cannot resume a prior P4-only checkpoint; set resume=False.')
        model.module.encoder.load_state_dict(state['student_encoder_state_dict'], strict=True); model.module.teacher.load_state_dict(state['teacher_encoder_state_dict'], strict=True); model.module.predictors.load_state_dict(state['predictors_state_dict'], strict=True)
        optimizer.load_state_dict(state['optimizer_state_dict']); scaler.load_state_dict(state['scaler_state_dict']); best_loss, global_step, start_epoch = state['best_loss'], state['global_step'], state['epoch'] + 1
        if rank == 0 and history_path.is_file():
            with history_path.open(newline='', encoding='utf-8') as handle: history = list(csv.DictReader(handle))
    steps_per_epoch, total_steps = len(loader), len(loader) * config['epochs']; warmup_steps = config['warmup_epochs'] * steps_per_epoch
    for epoch in range(start_epoch, config['epochs']):
        sampler.set_epoch(epoch); model.train(); started = time.perf_counter(); loss_sum = torch.zeros(1, device=device); count = torch.zeros(1, device=device); skipped = 0; health = None
        for batch_index, images in enumerate(loader):
            lr = learning_rate(config, global_step, total_steps, warmup_steps); decay = cosine_schedule(config['weight_decay_start'], config['weight_decay_end'], global_step, total_steps)
            for group_index, group in enumerate(optimizer.param_groups): group['lr'] = lr; group['weight_decay'] = decay if group_index == 0 else 0.0
            images = images.to(device, non_blocking=True); masks = sample_target_masks(images.shape[0], images.shape[-2] // config['p4_stride'], images.shape[-1] // config['p4_stride'], config, device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=config['amp']):
                predictions, targets, target_unions, _, _ = model(images, masks)
                p4_loss = F.smooth_l1_loss(predictions['p4'][target_unions['p4'].flatten(1)].float(), targets['p4'][target_unions['p4'].flatten(1)])
                p5_loss = F.smooth_l1_loss(predictions['p5'][target_unions['p5'].flatten(1)].float(), targets['p5'][target_unions['p5'].flatten(1)])
                loss = config['p4_loss_weight'] * p4_loss + config['p5_loss_weight'] * p5_loss
            if not torch.isfinite(loss): raise FloatingPointError(f'Non-finite loss at epoch {epoch + 1}, batch {batch_index + 1}: {loss.item()}')
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            gradients_finite = gradients_are_finite(model.parameters())
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), config['gradient_clip_norm']) if gradients_finite else torch.tensor(float('nan'), device=device)
            scale_before = scaler.get_scale(); scaler.step(optimizer); scaler.update(); step_skipped = not gradients_finite or scaler.get_scale() < scale_before
            if step_skipped:
                skipped += 1; consecutive_amp_skips += 1
                if consecutive_amp_skips > config['max_consecutive_amp_skips']: raise FloatingPointError('AMP skipped too many consecutive optimizer updates; reduce learning rate or inspect the generated checkpoint.')
            else:
                consecutive_amp_skips = 0; model.module.update_teacher(momentum_for_step(config, global_step, total_steps)); global_step += 1
            loss_sum += loss.detach() * images.shape[0]; count += images.shape[0]
        model.eval(); health = torch.zeros(4, device=device)
        if rank == 0:
            with torch.inference_mode():
                probe = next(iter(probe_loader)).to(device, non_blocking=True)
                probe_masks = sample_target_masks(probe.shape[0], probe.shape[-2] // config['p4_stride'], probe.shape[-1] // config['p4_stride'], config, device)
                probe_predictions, probe_targets, probe_unions, _, probe_teacher = model.module(probe, probe_masks)
                health = health_metrics(probe_teacher['p4'], probe_predictions['p4'], probe_targets['p4'], probe_unions['p4'])
        dist.broadcast(health, src=0); model.train()
        dist.all_reduce(loss_sum, op=dist.ReduceOp.SUM); dist.all_reduce(count, op=dist.ReduceOp.SUM)
        epoch_loss = (loss_sum / count).item(); health_values = {'feature_std': health[0].item(), 'mean_pairwise_cosine': health[1].item(), 'pairwise_cosine_p95': health[2].item(), 'prediction_target_cosine': health[3].item()}
        handoff_valid = health_values['feature_std'] >= config['min_feature_std'] and health_values['pairwise_cosine_p95'] < config['max_cosine_p95']
        if rank == 0:
            row = {'epoch': epoch + 1, 'loss': epoch_loss, 'learning_rate': lr, 'weight_decay': decay, 'ema_momentum': momentum_for_step(config, global_step, total_steps), 'amp_skipped_updates': skipped, 'seconds': time.perf_counter() - started, 'handoff_valid': handoff_valid, **health_values}; history.append(row)
            with history_path.open('w', newline='', encoding='utf-8') as handle: writer = csv.DictWriter(handle, fieldnames=row.keys()); writer.writeheader(); writer.writerows(history)
            state = save_state(output_dir, epoch, best_loss, global_step, model.module, optimizer, scaler, config, health_values)
            if epoch_loss < best_loss:
                torch.save(state, output_dir / 'lowest_loss_unvalidated.pt')
                if handoff_valid: best_loss = epoch_loss; state['best_loss'] = best_loss; torch.save(state, output_dir / 'best_ssl.pt')
            torch.save(state, output_dir / 'last_ssl.pt')
            print(f"Epoch {epoch + 1:02d}/{config['epochs']} | loss={epoch_loss:.5f} | feature_std={health_values['feature_std']:.5f} | p95_cos={health_values['pairwise_cosine_p95']:.5f} | valid={handoff_valid}")
        dist.barrier(device_ids=[local_rank])
    if rank == 0:
        if not (output_dir / 'best_ssl.pt').is_file(): raise RuntimeError('No non-collapsed checkpoint passed the image-only handoff guard; do not publish this run.')
        report = export_backbone(output_dir, config)
        manifest = {'method': 'I-JEPA', 'strict_label_free': True, 'ssl_images': len(paths), 'world_size': world_size, 'local_source_batch': config['per_gpu_batch'], 'global_source_batch': world_size * config['per_gpu_batch'], 'feature_grids': {'p4': f"{config['image_size'] // config['p4_stride']}x{config['image_size'] // config['p4_stride']}", 'p5': f"{config['image_size'] // config['p5_stride']}x{config['image_size'] // config['p5_stride']}"}, 'loss': '0.5 * P4 Smooth-L1 + 0.5 * P5 Smooth-L1', 'best_loss': best_loss, 'checkpoint_selection': 'lowest finite SSL loss that also passes the image-only anti-collapse guard', 'outputs': {'best_ssl': 'best_ssl.pt', 'last_ssl': 'last_ssl.pt', 'lowest_loss_unvalidated': 'lowest_loss_unvalidated.pt', 'history': 'history.csv', 'backbone': 'ijepa_best_backbone_yolo12m.pt', 'detector': 'ijepa_best_backbone_yolo12m_detector.pt', 'transfer_report': 'transfer_report.json'}, 'config': config}
        (output_dir / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8'); print(json.dumps(report, indent=2))
    dist.barrier(device_ids=[local_rank]); dist.destroy_process_group()
'''), encoding='utf-8')

RUNNER = OUTPUT_DIR / 'train_ijepa.py'
RUNNER.write_text("from pathlib import Path\nimport json\nfrom ijepa_core import train_distributed\nconfig = json.loads(Path(__file__).with_name('ijepa_config.json').read_text(encoding='utf-8'))\ntrain_distributed(config)\n", encoding='utf-8')
print(f'Wrote standalone core: {CORE_MODULE}')
print(f'Wrote DDP launcher: {RUNNER}')

In [ ]:
import random
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torchvision
import ultralytics
from PIL import Image
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

sys.path.insert(0, str(OUTPUT_DIR))
from ijepa_core import IMAGENET_MEAN, IMAGENET_STD, IJEPA, ImageOnlyDataset, YOLOPyramidEncoder, build_ijepa_transform, list_ssl_images, sample_target_masks

assert sys.version_info >= (3, 10), 'Python 3.10+ is required.'
assert tuple(map(int, torch.__version__.split('+')[0].split('.')[:2])) >= (2, 4), 'PyTorch 2.4+ is required.'
assert tuple(map(int, torchvision.__version__.split('+')[0].split('.')[:2])) >= (0, 19), 'Torchvision 0.19+ is required.'
assert tuple(map(int, ultralytics.__version__.split('.')[:3])) >= (8, 4, 96), 'Ultralytics 8.4.96+ is required.'
assert SSL_IMAGES.is_dir(), f'Attach AgroPest-12-PartB-Safe. Missing: {SSL_IMAGES}'
assert not any(SSL_IMAGES.glob('*.txt')), 'The SSL input must not contain text files.'
ssl_paths = list_ssl_images(SSL_IMAGES)
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, 'Select Kaggle GPU T4 x2 before training.'
gpu_names = [torch.cuda.get_device_name(index) for index in range(2)]
assert all('T4' in name for name in gpu_names), f'Expected two Kaggle T4 GPUs, found: {gpu_names}'
random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed); torch.cuda.manual_seed_all(cfg.seed)
torch.set_float32_matmul_precision('high'); sns.set_theme(style='whitegrid')
pd.Series({'Python': sys.version.split()[0], 'PyTorch': torch.__version__, 'Torchvision': torchvision.__version__, 'Ultralytics': ultralytics.__version__, 'GPU 0': gpu_names[0], 'GPU 1': gpu_names[1], 'SSL images': len(ssl_paths), 'label files read': 0, 'global source batch': cfg.per_gpu_batch * 2, 'P4 grid': f'{cfg.image_size // cfg.p4_stride}x{cfg.image_size // cfg.p4_stride}', 'P5 grid': f'{cfg.image_size // cfg.p5_stride}x{cfg.image_size // cfg.p5_stride}'})

## Input, Mask, And Finite-Loss Preflight

The original I-JEPA recipe uses random resized crops without colour distortion, blur, or horizontal flips. The preview shows the unmasked teacher image, the online image after grid-aligned target masking, and the sampled P4 target union. The preflight checks finite combined P4+P5 loss, online gradients, and target gradient isolation.

In [ ]:
preview_dataset = ImageOnlyDataset(ssl_paths, build_ijepa_transform(CONFIG, train=True))
images = torch.stack([preview_dataset[index] for index in random.Random(323).sample(range(len(preview_dataset)), 4)])
device = torch.device('cuda:0')
sanity_detector = ultralytics.YOLO(cfg.model_yaml).model.to(device)
sanity_encoder = YOLOPyramidEncoder(sanity_detector, cfg.p4_stride, cfg.p5_stride).to(device)
with torch.no_grad(): feature_dims = {level: value.shape[1] for level, value in sanity_encoder(torch.zeros(1, 3, cfg.image_size, cfg.image_size, device=device)).items()}
sanity_model = IJEPA(sanity_encoder, CONFIG, feature_dims).to(device)
masks = sample_target_masks(len(images), cfg.image_size // cfg.p4_stride, cfg.image_size // cfg.p4_stride, CONFIG, device)
union = masks.any(dim=1); pixel_mask = torch.nn.functional.interpolate(union.float().unsqueeze(1), size=(cfg.image_size, cfg.image_size), mode='nearest').bool()
mean, std = torch.tensor(IMAGENET_MEAN).view(3, 1, 1), torch.tensor(IMAGENET_STD).view(3, 1, 1)
fig, axes = plt.subplots(len(images), 3, figsize=(11, 3.5 * len(images)))
for row, image in enumerate(images):
    axes[row, 0].imshow((image * std + mean).clamp(0, 1).permute(1, 2, 0)); axes[row, 0].set_title('EMA teacher image')
    axes[row, 1].imshow((image.masked_fill(pixel_mask[row].cpu(), 0.0) * std + mean).clamp(0, 1).permute(1, 2, 0)); axes[row, 1].set_title('Online context image')
    axes[row, 2].imshow(union[row].cpu(), cmap='Reds', vmin=0, vmax=1); axes[row, 2].set_title('P4 target-block union')
    [axis.axis('off') for axis in axes[row]]
plt.tight_layout(); plt.show()
with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=cfg.amp):
    predictions, targets, target_unions, _, _ = sanity_model(images.to(device), masks)
    sanity_loss = cfg.p4_loss_weight * torch.nn.functional.smooth_l1_loss(predictions['p4'][target_unions['p4'].flatten(1)].float(), targets['p4'][target_unions['p4'].flatten(1)]) + cfg.p5_loss_weight * torch.nn.functional.smooth_l1_loss(predictions['p5'][target_unions['p5'].flatten(1)].float(), targets['p5'][target_unions['p5'].flatten(1)])
assert torch.isfinite(sanity_loss), f'Non-finite preflight loss: {sanity_loss.item()}'
sanity_loss.backward()
online_gradient = any(parameter.grad is not None and torch.isfinite(parameter.grad).all() for parameter in sanity_model.encoder.parameters())
teacher_gradient = any(parameter.grad is not None for parameter in sanity_model.teacher.parameters())
assert online_gradient and not teacher_gradient, 'Online encoder needs finite gradients and EMA teacher must have none.'
print({'p4_feature_dim': feature_dims['p4'], 'p5_feature_dim': feature_dims['p5'], 'finite_loss': sanity_loss.detach().item(), 'finite_online_gradient': online_gradient, 'teacher_has_gradient': teacher_gradient})
del sanity_model, sanity_detector; torch.cuda.empty_cache()

## Dual-GPU I-JEPA Training

The launcher uses NCCL, `DistributedSampler`, `DistributedDataParallel`, AMP, finite-loss and finite-gradient checks, gradient clipping, rank-zero-only writes, and resumable epoch checkpoints. Crucially, the EMA teacher, momentum schedule, and global-step counter advance only after a successful AMP optimizer update. A fixed image-only health check records target-feature variation and pairwise cosine similarity each epoch; a checkpoint is eligible for handoff only when it passes that guard.

In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2', str(RUNNER)], check=True)

## Training Curves And Checkpoint Handoff

SSL loss is not detection mAP. It is the equally weighted P4+P5 latent Smooth-L1 objective, so all transferable backbone stages receive gradients. `lowest_loss_unvalidated.pt` preserves the lowest-loss epoch for audit. `best_ssl.pt` is the lowest-loss checkpoint that also passes the image-only anti-collapse guard, preventing the previous near-zero-loss collapse from being published. The export enforces 100% key and shape coverage for the online backbone; teacher, predictors, optimizer, scheduler state, neck, and detection head are excluded.

In [ ]:
from matplotlib.ticker import MaxNLocator
history = pd.read_csv(OUTPUT_DIR / 'history.csv')
manifest = json.loads((OUTPUT_DIR / 'run_manifest.json').read_text(encoding='utf-8'))
transfer_report = json.loads((OUTPUT_DIR / 'transfer_report.json').read_text(encoding='utf-8'))
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
axes[0].plot(history['epoch'], history['loss'], marker='o', color='#1261A0'); axes[0].set(title='I-JEPA Latent Smooth-L1 Loss', xlabel='Epoch', ylabel='Loss')
axes[1].plot(history['epoch'], history['learning_rate'], marker='o', label='LR'); axes[1].plot(history['epoch'], history['weight_decay'], marker='o', label='Weight decay'); axes[1].set(title='Optimizer Schedules', xlabel='Epoch'); axes[1].legend()
axes[2].plot(history['epoch'], history['feature_std'], marker='o', label='Target feature std'); axes[2].axhline(cfg.min_feature_std, color='crimson', linestyle='--', label='Minimum valid std'); axes[2].set(title='Image-Only Feature Health', xlabel='Epoch'); axes[2].legend()
for axis in axes: axis.xaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'ijepa_training_curves.png', dpi=cfg.figure_dpi); plt.show()
pd.Series({'best SSL loss': manifest['best_loss'], 'checkpoint rule': manifest['checkpoint_selection'], 'best SSL checkpoint': OUTPUT_DIR / 'best_ssl.pt', 'resumable checkpoint': OUTPUT_DIR / 'last_ssl.pt', 'backbone checkpoint': OUTPUT_DIR / 'ijepa_best_backbone_yolo12m.pt', 'detector-compatible checkpoint': OUTPUT_DIR / 'ijepa_best_backbone_yolo12m_detector.pt', 'backbone coverage': f"{transfer_report['backbone_key_coverage']:.1%}"})

## Image-Only Representation Diagnostics

t-SNE and cosine retrieval use a fixed sample from `ssl_pool/images` only. They are exploratory image-level diagnostics, not class metrics or evidence of detection quality. Describe visible shared morphology, leaf texture, framing, colour, or background in the retrieval panels, and reserve detection claims for Notebook 03b.

In [ ]:
best_state = torch.load(OUTPUT_DIR / 'best_ssl.pt', map_location='cpu', weights_only=False)
diagnostic_detector = ultralytics.YOLO(cfg.model_yaml).model
diagnostic_encoder = YOLOPyramidEncoder(diagnostic_detector, cfg.p4_stride, cfg.p5_stride)
diagnostic_encoder.load_state_dict(best_state['student_encoder_state_dict'], strict=True)
diagnostic_encoder = diagnostic_encoder.to(device).eval()
diagnostic_paths = sorted(random.Random(cfg.seed).sample(ssl_paths, min(cfg.diagnostic_images, len(ssl_paths))))
evaluation_transform = build_ijepa_transform(CONFIG, train=False)

class FeatureDataset(Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image: return evaluation_transform(image.convert('RGB')), self.paths[index].name

feature_loader = DataLoader(FeatureDataset(diagnostic_paths), batch_size=cfg.diagnostic_batch_size, shuffle=False, num_workers=cfg.workers, pin_memory=True)
features, names = [], []
with torch.inference_mode():
    for images, batch_names in feature_loader:
        features.append(torch.nn.functional.normalize(diagnostic_encoder(images.to(device, non_blocking=True))['p4'].mean((2, 3)), dim=1).cpu())
        names.extend(batch_names)
feature_matrix = torch.cat(features)
assert torch.isfinite(feature_matrix).all()
pairwise = feature_matrix @ feature_matrix.T
off_diagonal = pairwise[~torch.eye(len(feature_matrix), dtype=torch.bool)]
diagnostics = pd.Series({'diagnostic images': len(names), 'feature dimensions': feature_matrix.shape[1], 'mean L2 norm': feature_matrix.norm(dim=1).mean().item(), 'mean feature std': feature_matrix.std(dim=0).mean().item(), 'mean off-diagonal cosine': off_diagonal.mean().item(), 'off-diagonal cosine std': off_diagonal.std().item()})
assert diagnostics['mean feature std'] >= cfg.min_feature_std, 'Collapsed constant representations detected; do not publish this checkpoint.'
display(diagnostics)

perplexity = min(cfg.tsne_perplexity, max(5.0, (len(feature_matrix) - 1) / 3))
coordinates = TSNE(n_components=2, perplexity=perplexity, learning_rate='auto', init='pca', max_iter=cfg.tsne_iterations, random_state=cfg.seed).fit_transform(feature_matrix.numpy())
fig, axis = plt.subplots(figsize=(9, 7)); axis.scatter(coordinates[:, 0], coordinates[:, 1], s=15, alpha=0.7, color='#7c3aed'); axis.set(title='t-SNE: I-JEPA YOLOv12m SSL-Pool Features', xlabel='t-SNE 1', ylabel='t-SNE 2')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'ijepa_tsne_ssl_pool.png', dpi=cfg.figure_dpi); plt.show()
pd.DataFrame({'filename': names, 'tsne_1': coordinates[:, 0], 'tsne_2': coordinates[:, 1]}).to_csv(OUTPUT_DIR / 'ssl_pool_embeddings_tsne.csv', index=False)

query_indices = random.Random(cfg.seed).sample(range(len(diagnostic_paths)), cfg.retrieval_queries)
fig, axes = plt.subplots(cfg.retrieval_queries, cfg.retrieval_neighbors + 1, figsize=(3.2 * (cfg.retrieval_neighbors + 1), 3.2 * cfg.retrieval_queries)); retrieval_rows = []
for row, query_index in enumerate(query_indices):
    similarities = feature_matrix @ feature_matrix[query_index]
    neighbors = [index for index in torch.topk(similarities, k=cfg.retrieval_neighbors + 1).indices.tolist() if index != query_index][:cfg.retrieval_neighbors]
    for column, index in enumerate([query_index] + neighbors):
        with Image.open(diagnostic_paths[index]) as image: axes[row, column].imshow(image.convert('RGB'))
        axes[row, column].set_title('Query' if column == 0 else f'cos={similarities[index]:.3f}', fontsize=9); axes[row, column].axis('off')
        retrieval_rows.append({'query': diagnostic_paths[query_index].name, 'rank': column, 'match': diagnostic_paths[index].name, 'cosine_similarity': float(similarities[index])})
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'ijepa_nearest_neighbors.png', dpi=cfg.figure_dpi); plt.show()
pd.DataFrame(retrieval_rows).to_csv(OUTPUT_DIR / 'ijepa_nearest_neighbors.csv', index=False)
print('After execution, describe only visible image-level similarity in retrieved neighbours; reserve all detection-performance claims for Notebook 03b.')

## Publication And Viva Notes

Publish the complete `/kaggle/working/partB_03a_ijepa_yolo12m/` directory as a Kaggle dataset. Notebook 03b must attach it and use `ijepa_best_backbone_yolo12m.pt` together with `transfer_report.json`.

- The online/context encoder receives pixel-masked images and is optimized by AdamW.
- The target encoder receives complete images, receives no gradients, and follows the online encoder by EMA after successful updates only.
- Separate predictors replace target content with learned mask tokens, then add distinct 2D positions; they predict teacher P4 and P5 latents, not pixels.
- Four target blocks are sampled on the 16x16 P4 grid and pooled onto P5. Their union is excluded from visible context, so prediction cannot copy target content.
- Both latent losses are equally weighted. This trains every transferable backbone stage before the online YOLO backbone transfers with 100% key and shape coverage.
- A falling I-JEPA loss alone is not proof of useful features. The per-epoch feature-variance and cosine guard prevents publishing the collapse observed with the prior library.